# Section A and the attribution maps

This notebook did section A (the summer checkpoints scored again, Nov 2, 2026) and the attribution
maps (A2, Nov 3, 2026). Both are done, their results are on Drive in `results_v2/` and their
output is kept in `docs/runs/`.

**Section B (retraining the ViT) is not run from here anymore.** Each seed has its own notebook in
`notebooks/runs/` (`B_vit_v2_seed42.ipynb`, `..43`, `..44`), so every seed has its own file and its
own record, and a disconnect only means running the same notebook again. See `docs/runs/README.md`.

The "Could not find cuda drivers" lines in the output come from TensorFlow, which Colab
loads but we don't use. PyTorch still uses the GPU (see the `GPU:` line in the first cell).


In [1]:
# we mount Drive (where the NHATS data and the checkpoints are) and get the latest code
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/cdt-alzheimer-screening'
if not os.path.exists(REPO):
    !git clone -q https://github.com/wiambenadder/cdt-alzheimer-screening.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!git log --oneline -1

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, switch the runtime to GPU')

Mounted at /content/drive
/content/cdt-alzheimer-screening
22ab4aa (HEAD -> main, origin/main, origin/HEAD) one notebook per seed, continue a cut run, and a record for every run
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# the settings, change these if your Drive folder is somewhere else
DRIVE_DATA = '/content/drive/MyDrive/cdt-data'

RUN_SECTION_A = False   # re-score the v1 checkpoints with CIs (done Nov 2, results on Drive)
RUN_MAPS = False         # attribution maps of the v1 ViT, drawn on the model input (a few min)
RUN_SECTION_B = False   # section B moved to notebooks/runs/ (one notebook per seed)
B_SEEDS = ''
WORKERS = os.cpu_count()  # reading the TIFFs is the slow part, so we use every CPU

os.environ['CDT_DATA_DIR'] = DRIVE_DATA                       # splits.csv is saved here
os.environ['CDT_MODELS_DIR'] = f'{DRIVE_DATA}/models'
os.environ['CDT_RESULTS_DIR'] = f'{DRIVE_DATA}/results_v2'    # on Drive so nothing is lost
os.makedirs(os.environ['CDT_RESULTS_DIR'], exist_ok=True)
print('checkpoints on Drive:', sorted(os.listdir(os.environ['CDT_MODELS_DIR'])))

checkpoints on Drive: ['abl_frozen_aug_best.pt', 'abl_frozen_noaug_best.pt', 'abl_unfrozen_aug_best.pt', 'abl_unfrozen_noaug_best.pt', 'effb0_ft_best.pt', 'vgg16_frozen_baseline_best.pt', 'vgg16_ft_best.pt', 'vit_b16_ft_best.pt', 'vit_b16_v2_s42_best.pt', 'vit_b16_v2_s43_best.pt']


In [ ]:
# we copy the TIFFs to the local disk first, reading 59k files from Drive during training is too slow
import shutil, time
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

LOCAL = Path('/content/nhats_local')
LOCAL.mkdir(exist_ok=True)
RAW = Path(f'{DRIVE_DATA}/nhats_raw')
t0 = time.time()
for rf in sorted(RAW.iterdir()):
    if not rf.is_dir():
        continue
    dst = LOCAL / rf.name
    if dst.exists() and any(dst.glob('*.tif')):
        continue
    print(f'[copy] {rf.name}')
    shutil.copytree(rf, dst, dirs_exist_ok=True)
print(f'staged in {(time.time() - t0) / 60:.1f} min')

labels = pd.read_csv(f'{DRIVE_DATA}/labels.csv')
labels['image_path'] = labels['image_path'].str.replace(f'{DRIVE_DATA}/nhats_raw/', str(LOCAL) + '/', regex=False)

# a copy that was stopped in the middle (or a disconnect) leaves half-written files, and training
# crashes when it reaches one. So we check every file: if it's missing or a different size than
# on Drive we copy it again, then we open it fully. Section A never saw this because it only
# reads the val and test clocks, training reads all of them.
def check(path):
    local = Path(path)
    drive = RAW / local.relative_to(LOCAL)
    try:
        if not local.exists() or local.stat().st_size != drive.stat().st_size:
            shutil.copy2(drive, local)
        with Image.open(local) as im:
            im.load()
        return None
    except Exception:
        return path

t0 = time.time()
with ThreadPoolExecutor(16) as pool:
    bad = [p for p in pool.map(check, labels['image_path']) if p]
print(f'checked {len(labels):,} files in {(time.time() - t0) / 60:.1f} min')

# whatever is still broken is broken on Drive too, so we leave those clocks out.
# the split is by participant, so this doesn't move anybody between train, val and test.
# we only print how many (the file names are participant IDs)
if bad:
    Path(f'{DRIVE_DATA}/broken_files.txt').write_text('\n'.join(bad))
    labels = labels[~labels['image_path'].isin(bad)]
    print(f'{len(bad)} files are broken even on Drive and are left out '
          f'(list in cdt-data/broken_files.txt)')
else:
    print('all files open fine')

LABELS = '/content/labels_local.csv'
labels.to_csv(LABELS, index=False)
print(f'{len(labels):,} labels -> {LABELS}')

[copy] round_01
[copy] round_02


## A. Re-score the v1 checkpoints

The v1 models were trained on the v1 crop, so we score them with `--crop-method projection`.
The first run also writes `splits.csv` to Drive, every run after that reuses it.

In [ ]:
if RUN_SECTION_A:
    v1 = {'vit_b16': 'vit_b16_ft', 'vgg16': 'vgg16_ft', 'efficientnet_b0': 'effb0_ft'}
    for model_name, ckpt_name in v1.items():
        ckpt = f"{os.environ['CDT_MODELS_DIR']}/{ckpt_name}_best.pt"
        if not os.path.exists(ckpt):
            print(f'[skip] {ckpt} not found')
            continue
        !python -m src.run --models {model_name} --evaluate-only --crop-method projection \
            --checkpoint {ckpt} --run-tag _v1 --labels {LABELS} --num-workers {WORKERS}
    !python -m src.run --compare vit_b16_v1_s42 vgg16_v1_s42

## A2. Attribution maps of the v1 ViT

The v1 figure drew the heatmap on the uncropped page. This one draws it on the exact
image the model sees, one example per score, so the picture can back up "the model looks
at the clock face and the hands". It uses the test predictions saved by section A.

In [ ]:
if RUN_MAPS:
    import pandas as pd
    from src.models import get_model
    from src.utils import load_checkpoint, get_device
    from src.augmentation import build_eval_transform
    from src.gradcam import plot_attribution_grid
    from IPython.display import Image as ShowImage, display

    pred = pd.read_csv(f"{os.environ['CDT_RESULTS_DIR']}/vit_b16_v1_s42/predictions_test.csv")
    device = get_device()
    model, _, _ = get_model('vit_b16', freeze_backbone=False)
    model = load_checkpoint(model, f"{os.environ['CDT_MODELS_DIR']}/vit_b16_ft_best.pt", device).to(device).eval()

    out = f"{os.environ['CDT_RESULTS_DIR']}/vit_b16_v1_s42/attribution_maps.png"
    # the v1 checkpoint was trained on the v1 crop, so the maps use it too
    plot_attribution_grid(model, pred, pred['y_true'].values, pred['y_pred'].values,
                          build_eval_transform(224, crop_method='projection'), device, save_path=out)
    print('saved', out)
    display(ShowImage(out, width=500))

## B. Retrain ViT-B/16 with the v2 preprocessing

Moved to `notebooks/runs/B_vit_v2_seed42.ipynb`, `B_vit_v2_seed43.ipynb` and `B_vit_v2_seed44.ipynb`.
This cell is kept off so nothing is trained twice.

In [ ]:
if RUN_SECTION_B:
    !python -m src.run --models vit_b16 --seeds {B_SEEDS} --select-metric val_kappa \
        --run-tag _v2 --labels {LABELS} --num-workers {WORKERS}
    # each new seed against the summer ViT, on the same test clocks
    if os.path.exists(f"{os.environ['CDT_RESULTS_DIR']}/vit_b16_v1_s42"):
        for seed in B_SEEDS.split():
            if os.path.exists(f"{os.environ['CDT_RESULTS_DIR']}/vit_b16_v2_s{seed}/predictions_test.csv"):
                !python -m src.run --compare vit_b16_v2_s{seed} vit_b16_v1_s42

## Summary to paste back in the chat

Only aggregate numbers, no participant IDs, so it is fine to share.
For the full side-by-side table and the figures (training curves of every seed, confusion
matrices), open `notebooks/06_results_report.ipynb`.

In [ ]:
# this cell works on its own, even after a disconnect (it only reads the results on Drive)
import os, json, glob
import numpy as np
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
RESULTS = os.environ.get('CDT_RESULTS_DIR', '/content/drive/MyDrive/cdt-data/results_v2')

rows, v2 = [], []
for f in sorted(glob.glob(f"{RESULTS}/*/metrics.json")):
    m = json.load(open(f))
    k, ci = m['argmax']['quadratic_kappa'], m['argmax_ci']['quadratic_kappa']
    kt, cit = m['tuned']['quadratic_kappa'], m['tuned_ci']['quadratic_kappa']
    rows.append(f"{m['run_name']:24s} n_test={m['n_test']}  kappa={k:.3f} [{ci['lo']:.3f}, {ci['hi']:.3f}]  "
                f"tuned={kt:.3f} [{cit['lo']:.3f}, {cit['hi']:.3f}]  acc={m['argmax']['accuracy']:.3f}  "
                f"macroF1={m['argmax']['macro_f1']:.3f}")
    if m['run_name'].startswith('vit_b16_v2_s'):
        v2.append((k, kt))
for f in sorted(glob.glob(f"{RESULTS}/compare_*.json")):
    c = json.load(open(f))
    rows.append(f"{Path(f).stem}: diff={c['diff']:+.3f} [{c['diff_lo']:+.3f}, {c['diff_hi']:+.3f}]")
if len(v2) > 1:
    k, kt = np.array(v2).T
    rows.append(f"ViT v2 over {len(v2)} seeds: kappa {k.mean():.3f} (sd {k.std(ddof=1):.3f})  "
                f"tuned {kt.mean():.3f} (sd {kt.std(ddof=1):.3f})")
print('\n'.join(rows) if rows else 'no results yet')
# the v1 test set had 8,938 clocks, if n_test is different the split is not the same as v1